# NER Encoder Training - ViPubmedDeBERTa

**Môi trường:** Kaggle GPU (T4 x2 hoặc P100), Internet ON

**Mục tiêu:** Train encoder BIO cho nhãn SYM_DIS (từ PhoNER + ViMQ)

**Đầu ra:** Model weights + metrics để upload thành Kaggle Dataset

In [ ]:
# Cell 1: Install dependencies
!pip install -q transformers datasets seqeval pyvi accelerate

In [ ]:
# Cell 2: Clone code repository
import os

# TODO: Replace with your actual repo URL
REPO_URL = "https://github.com/YOUR_USERNAME/medical-ner-vietnamese.git"

if not os.path.exists('medical-ner-vietnamese'):
    !git clone {REPO_URL}

os.chdir('medical-ner-vietnamese')
!pwd

In [ ]:
# Cell 3: Download and convert datasets
import sys
sys.path.append('src')

from data_prep.convert_phoner import convert_all_phoner
from data_prep.convert_vimq import convert_all_vimq
import json

print("Converting PhoNER...")
phoner_data = convert_all_phoner()

print("\nConverting ViMQ...")
vimq_data = convert_all_vimq()

# Merge datasets
merged_data = {}
for split in ['train', 'dev', 'test']:
    merged_data[split] = phoner_data[split] + vimq_data[split]
    print(f"{split}: {len(merged_data[split])} examples")

In [ ]:
# Cell 4: Prepare dataset for training
from transformers import AutoTokenizer
from datasets import Dataset, DatasetDict

# Model checkpoint
MODEL_NAME = "manhtt-079/vipubmed-deberta-base"

# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Verify tokenizer is fast (has word_ids())
test_encoding = tokenizer(["test"], is_split_into_words=True)
try:
    test_encoding.word_ids(0)
    print("✓ Tokenizer is FAST - word_ids() available")
except:
    raise ValueError("❌ Tokenizer is NOT FAST - cannot use word_ids()!")

# Label mapping
label_list = ['O', 'B-SYM_DIS', 'I-SYM_DIS']
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print(f"\nLabel mapping: {label2id}")

# Convert to HF Dataset
dataset_dict = {}
for split in ['train', 'dev']:
    dataset_dict[split] = Dataset.from_list(merged_data[split])

datasets = DatasetDict(dataset_dict)
print(f"\nDatasets: {datasets}")

In [ ]:
# Cell 5: Tokenization with label alignment

def tokenize_and_align_labels(examples):
    """
    Tokenize words and align labels to subword tokens.
    
    Rules:
    - First subword of a word: keep label
    - Subsequent subwords: -100 (ignored in loss)
    - Special tokens: -100
    """
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True,
        max_length=256,
        padding=False  # Will pad in data collator
    )
    
    labels = []
    for i, label_seq in enumerate(examples['labels']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        label_ids = []
        previous_word_idx = None
        
        for word_idx in word_ids:
            # Special token
            if word_idx is None:
                label_ids.append(-100)
            # First subword of word
            elif word_idx != previous_word_idx:
                label = label_seq[word_idx]
                label_ids.append(label2id[label])
            # Subsequent subword
            else:
                label_ids.append(-100)
            
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

# Apply tokenization
print("Tokenizing datasets...")
tokenized_datasets = datasets.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=datasets['train'].column_names
)

print(f"\nTokenized: {tokenized_datasets}")

In [ ]:
# Cell 6: Load model and setup training
from transformers import (
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import numpy as np
from seqeval.metrics import classification_report, f1_score

# Load model
print(f"Loading model: {MODEL_NAME}")
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
    classifier_dropout=0.2
)

print(f"✓ Model loaded: {model.num_parameters():,} parameters")

# Data collator
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True
)

In [ ]:
# Cell 7: Metrics

def compute_metrics(pred):
    """
    Compute seqeval metrics.
    """
    predictions, labels = pred
    predictions = np.argmax(predictions, axis=2)
    
    # Remove ignored index (-100) and convert to labels
    true_labels = []
    true_predictions = []
    
    for prediction, label in zip(predictions, labels):
        true_label = []
        true_pred = []
        for p, l in zip(prediction, label):
            if l != -100:
                true_label.append(id2label[l])
                true_pred.append(id2label[p])
        true_labels.append(true_label)
        true_predictions.append(true_pred)
    
    # Compute metrics
    micro_f1 = f1_score(true_labels, true_predictions, average='micro')
    
    return {
        'f1': micro_f1
    }

In [ ]:
# Cell 8: Training arguments (từ paper)

training_args = TrainingArguments(
    output_dir='/kaggle/working/ner_encoder_checkpoints',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.015,
    warmup_ratio=0.05,
    adam_epsilon=1e-9,
    max_grad_norm=1.0,
    
    # Evaluation
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    greater_is_better=True,
    
    # Logging
    logging_steps=100,
    report_to='none',
    
    # Other
    seed=42,
    fp16=True,  # Mixed precision
    dataloader_num_workers=2,
    save_total_limit=2
)

print("Training arguments:")
print(training_args)

In [ ]:
# Cell 9: Initialize Trainer and train

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['dev'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

print("Starting training...")
train_result = trainer.train()

print("\n" + "="*60)
print("Training completed!")
print("="*60)
print(train_result)

In [ ]:
# Cell 10: Final evaluation on dev set

print("Running final evaluation...")
metrics = trainer.evaluate()

print("\n" + "="*60)
print("FINAL METRICS")
print("="*60)
for key, value in metrics.items():
    print(f"{key}: {value}")

# Detailed report
predictions, labels, _ = trainer.predict(tokenized_datasets['dev'])
predictions = np.argmax(predictions, axis=2)

true_labels = []
true_predictions = []

for prediction, label in zip(predictions, labels):
    true_label = []
    true_pred = []
    for p, l in zip(prediction, label):
        if l != -100:
            true_label.append(id2label[l])
            true_pred.append(id2label[p])
    true_labels.append(true_label)
    true_predictions.append(true_pred)

print("\n" + classification_report(true_labels, true_predictions))

In [ ]:
# Cell 11: Save model for Kaggle Dataset
import json

output_dir = '/kaggle/working/ner_encoder'
os.makedirs(output_dir, exist_ok=True)

# Save model
print(f"Saving model to {output_dir}...")
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

# Save label mapping
label_map = {
    'label2id': label2id,
    'id2label': {str(k): v for k, v in id2label.items()}
}
with open(f'{output_dir}/label_map.json', 'w') as f:
    json.dump(label_map, f, indent=2, ensure_ascii=False)

# Save metrics
metrics_summary = {
    'model': MODEL_NAME,
    'dev_f1': metrics['eval_f1'],
    'num_labels': len(label_list),
    'training_examples': len(tokenized_datasets['train']),
    'dev_examples': len(tokenized_datasets['dev'])
}
with open(f'{output_dir}/metrics.json', 'w') as f:
    json.dump(metrics_summary, f, indent=2)

print("\n" + "="*60)
print("✓ Model saved successfully!")
print("="*60)
print(f"Location: {output_dir}")
print(f"Dev F1: {metrics['eval_f1']:.4f}")
print("\nNext steps:")
print("1. Download this folder from Kaggle output")
print("2. Create new Kaggle Dataset and upload the folder")
print("3. Use the dataset in inference notebook")